In [1]:
import joblib
import pandas as pd
from sklearn import set_config
from tempfile import TemporaryDirectory
from sklearn.svm import LinearSVC
from sklearn.utils.class_weight import compute_sample_weight
import warnings

In [2]:
target_column = "health_condition"

In [3]:
X_train = pd.read_csv("../data/intermediate/train_features_oheencoded_imputed_scaled.csv")
X_valid = pd.read_csv("../data/intermediate/valid_features_oheencoded_imputed_scaled.csv")
X_test = pd.read_csv("../data/intermediate/test_features_oheencoded_imputed_scaled.csv")

y_train = pd.read_csv("../data/intermediate/train_labels.csv")
y_valid = pd.read_csv("../data/intermediate/valid_labels.csv")

X_train.info()

<class 'pandas.DataFrame'>
RangeIndex: 552070 entries, 0 to 552069
Data columns (total 39 columns):
 #   Column                             Non-Null Count   Dtype  
---  ------                             --------------   -----  
 0   sleep_duration                     552070 non-null  float64
 1   heart_rate                         552070 non-null  float64
 2   bmi                                552070 non-null  float64
 3   calorie_expenditure                552070 non-null  float64
 4   step_count                         552070 non-null  float64
 5   exercise_duration                  552070 non-null  float64
 6   water_intake                       552070 non-null  float64
 7   stress_level                       552070 non-null  float64
 8   sleep_quality                      552070 non-null  float64
 9   physical_activity_level            552070 non-null  float64
 10  smoking_alcohol                    552070 non-null  float64
 11  calorie_expenditure_per_step       552070 non-null

In [4]:
X = pd.concat([X_train, X_valid], axis=0)
y = pd.concat([y_train, y_valid], axis=0)

In [5]:
df_submission = pd.read_csv("../data/sample_submission.csv")
df_submission.info()

<class 'pandas.DataFrame'>
RangeIndex: 295753 entries, 0 to 295752
Data columns (total 2 columns):
 #   Column            Non-Null Count   Dtype
---  ------            --------------   -----
 0   id                295753 non-null  int64
 1   health_condition  295753 non-null  str  
dtypes: int64(1), str(1)
memory usage: 4.5 MB


In [6]:
train_sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
valid_sample_weight = compute_sample_weight(class_weight="balanced", y=y_valid)
y_sample_weight = compute_sample_weight(class_weight="balanced", y=y)

model = LinearSVC(loss='squared_hinge', class_weight='balanced', max_iter=1000)
model.fit(X_train, y_train, sample_weight=train_sample_weight)

joblib.dump(model, '../models/linearsvc_80.pkl')

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['../models/linearsvc_80.pkl']

In [7]:
from sklearn.metrics import balanced_accuracy_score

y_pred = model.predict(X_valid)

val_score = balanced_accuracy_score(y_valid, y_pred, sample_weight=valid_sample_weight)
print("Validation Balanced Accuracy:", val_score)

Validation Balanced Accuracy: 0.8543595075890534


In [8]:
model = LinearSVC(loss='squared_hinge', class_weight='balanced', max_iter=1000)
model.fit(X, y, sample_weight=y_sample_weight)

joblib.dump(model, '../models/linearsvc_100.pkl')

c:\Users\abhin\.pyenv\pyenv-win\versions\3.14.2\Lib\site-packages\sklearn\utils\validation.py:1365: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


['../models/linearsvc_100.pkl']

In [9]:
y_pred = model.predict(X_test)

df_submission[target_column] = y_pred
df_submission[target_column] = df_submission[target_column].replace({0:'unhealthy', 1:'at-risk', 2: 'fit'})

df_submission.to_csv('../results/linearsvc_baseline.csv', index=False)
df_submission

,id,health_condition
0,690088,unhealthy
1,690089,unhealthy
2,690090,fit
3,690091,at-risk
4,690092,unhealthy
...,...,...
295748,985836,fit
295749,985837,unhealthy
295750,985838,unhealthy
295751,985839,at-risk
